In [1]:
%load_ext autoreload
%autoreload 2

# About this Notebook
This notebook is about testing the task loading and implementation as well as the PyTorch Dataset and DataLoader objects.

In [ ]:
import os
import torch
import logging
import matplotlib
import pandas as pd
import matplotlib.pyplot as plt

import src.utils.utils as utils
import src.tasks.taskloader as taskloader
import src.data.dataloader as dataloader
import src.train.training as training

from typing import List
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image, pil_to_tensor

from src.data.dataloader import get_metadata_df
from src.utils.utils import setup_logging
from src.dataset.dataset import NeuralizerOCTDataset
from src.dataset.sampler import WeightedTaskSampler

# Set to logging.DEBUG for more information
setup_logging(loglevel=logging.INFO)
logger = logging.getLogger(__name__)

# Set logging level of external libraries to INFO to avoid cluttering
logging.getLogger('PIL').setLevel(logging.INFO)
logging.getLogger('matplotlib').setLevel(logging.INFO)

os.chdir("../..")
print(os.getcwd())

# Load CFG file
CFG: dict = utils.load_config("configs/config.yaml")

# Matplotlib settings
matplotlib.rcParams['mathtext.fontset'] = 'cm'  
matplotlib.rcParams['font.family'] = 'STIXGeneral'

# Load Metadata

In [ ]:
df = get_metadata_df(path="data/metadata.csv")
df.head(5)

# Test Base Tasks

In [4]:
from src.tasks.base_task import BaseTask
from src.tasks.binary_segmentations import DUKEFluidSegmentation, UMNFluidSegmentation
from src.tasks.multi_segmentations import DUKELayerSegmentation
from src.tasks.synthetic_tasks import ImageRotation, VerticalFlip

from torchvision.transforms import v2

# Test Dataloader

In [ ]:
metadata_df: pd.DataFrame = dataloader.get_metadata_df(CFG["dataloader"]["metadata_location"])
enrichment_df: pd.DataFrame = dataloader.get_metadata_df(CFG["dataloader"]["enrichment_location"])
tasks: List[BaseTask] = taskloader.get_tasks(df=metadata_df, enrichment_df=enrichment_df, CFG=CFG["tasks"])

In [ ]:
train_dataloader, val_dataloader, test_dataloader = training.get_dataloaders(CFG=CFG, tasks=tasks)

In [ ]:
# Get a sample from the dataset
x, y, ctx_in, ctx_out, lossfun, class_names, task_names = next(iter(train_dataloader)) 

# Print information
print("x shape: ", x.shape)
print("y shape: ", y.shape)
print("ctx_in shape: ", ctx_in.shape)
print("ctx_out shape: ", ctx_out.shape)
print("loss function(s): ", lossfun)
print("class name: ", class_names)
print("task name: ", task_names)

# Visualize Input x and Output y

In [ ]:
# Specify which batch to visualize
BATCH_IDX = 1

def plot_input_output(x: torch.Tensor, y: torch.Tensor):
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(4, 5))
    axes[0].set_title("Input x")
    axes[0].imshow(to_pil_image(x))
    axes[1].set_title("Output y")
    axes[1].imshow(to_pil_image(y))
    fig.tight_layout()

if x.ndim > 3:
    plot_input_output(x[BATCH_IDX], y[BATCH_IDX])
else:
    plot_input_output(x, y)

# Visualize Context C

In [ ]:
from src.visualizations.contextvizualizer import plot_context_set, plot_context_set_overlay

plot_context_set(ctx_in[BATCH_IDX])
plot_context_set(ctx_out[BATCH_IDX])
#plot_context_set_overlay(ctx_in[BATCH_IDX], ctx_out[BATCH_IDX])

________
# Visualize ALL Train Tasks (seen during Training)

In [10]:
from src.visualizations.contextvizualizer import plot_all_tasks

#plot_all_tasks(CFG, tasks=tasks, context_set_size=8)

# Visualize ALL Test Tasks (not seen during Training)

In [ ]:
test_tasks: List[BaseTask] = taskloader.get_test_tasks(df=metadata_df, enrichment_df=enrichment_df, CFG=CFG["tasks"])
#plot_all_tasks(CFG, tasks=test_tasks, context_set_size=8)

In [102]:
#fig.savefig("assets/unseen-generative-tasks-examples.png", bbox_inches='tight', dpi=256)